In [6]:
features_manter = [
    # --- Volumetria Bruta (Tamanho e Quantidade de Pacotes) ---
    'Flow Duration', 
    'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    
    # --- Estatísticas de Payload (Assimetria de Tamanho) ---
    'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std',
    'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std',
    'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance',
    'Average Packet Size', 'Avg Fwd Segment Size', 'Avg Bwd Segment Size',
    
    # --- Taxas e Cadência Temporal (Velocidade e Vazão) ---
    'Flow Bytes/s', 'Flow Packets/s', 
    'Fwd Packets/s', 'Bwd Packets/s',
    
    # --- Comportamento de Intervalo (IAT - Inter-Arrival Time) ---
    'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    
    # --- Estrutura Básica do Cabeçalho e Direção ---
    'Fwd Header Length', 'Bwd Header Length', 'Down/Up Ratio', 'act_data_pkt_fwd',
    
    # --- Métricas de Atividade de Conexão ---
    'Active Mean', 'Active Std', 'Active Max', 'Active Min', 
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min',
    
    # --- Alvo ---
    'Label'
]

In [ ]:
import os
import gc
import pandas as pd
import numpy as np

# --- Configuração de Caminhos ---
path_cic_udp = r"C:\Users\xrafa\OneDrive - Universidade Federal de Uberlândia\Nova pasta\Pesquisas\f(ANN)\ANN\notebooks\redes\udp\UDP.csv"
path_cic_lag = r"C:\Users\xrafa\OneDrive - Universidade Federal de Uberlândia\Nova pasta\Pesquisas\f(ANN)\ANN\notebooks\redes\udp_lag\UDPLag.csv"
path_unsw_folder = r"C:\Users\xrafa\Downloads\archive\\"

path_saida_cic = r"C:\Users\xrafa\Downloads\archive\CIC_Misto_UDP_Treino.csv"
path_saida_unsw = r"C:\Users\xrafa\Downloads\archive\UNSW_UDP_Validacao.csv"

# Garantir que trabalhamos apenas com as colunas que a sua rede precisa
features_finais_cic = [col for col in features_manter]
CHUNK_SIZE = 50000 
MAX_AMOSTRAS_POR_CHUNK = 2000

# ==============================================================================
# --- PARTE 1: PROCESSAMENTO DO CIC DDOS (TREINO MISTO UDP) ---
# ==============================================================================
chunks_treino = []

def processar_cic_streaming(path_arquivo, nome_log):
    print(f"-> Lendo blocos do arquivo CIC [{nome_log}]...")
    dtype_dict = {col: np.float32 for col in features_finais_cic if col != 'Label'}
    for chunk in pd.read_csv(path_arquivo, chunksize=CHUNK_SIZE, dtype=dtype_dict, low_memory=False):
        chunk.columns = chunk.columns.str.strip()
        valid_cols = [col for col in features_finais_cic if col in chunk.columns]
        chunk_f = chunk[valid_cols].copy()
        
        if 'Label' in chunk_f.columns:
            chunk_f['Label'] = chunk_f['Label'].apply(lambda x: 1 if 'BENIGN' not in str(x) else 0).astype(np.int8)
        
        chunk_f.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk_f.fillna(0.0, inplace=True)
        
        df_b = chunk_f[chunk_f['Label'] == 0]
        df_a = chunk_f[chunk_f['Label'] == 1]
        n_b, n_a = min(MAX_AMOSTRAS_POR_CHUNK, len(df_b)), min(MAX_AMOSTRAS_POR_CHUNK, len(df_a))
        
        if n_b > 0 and n_a > 0:
            chunks_treino.append(pd.concat([df_b.sample(n=n_b, random_state=42), df_a.sample(n=n_a, random_state=42)], ignore_index=True))
        del chunk, chunk_f, df_b, df_a
        gc.collect()

processar_cic_streaming(path_cic_udp, "UDP Puro")
processar_cic_streaming(path_cic_lag, "UDP Lag")

print("Unificando e balanceando CIC DDoS...")
df_cic_misto = pd.concat(chunks_treino, ignore_index=True)
del chunks_treino
gc.collect()

qtd_final_cic = min(100000, df_cic_misto['Label'].value_counts().min())
df_cic_final = pd.concat([
    df_cic_misto[df_cic_misto['Label'] == 0].sample(n=qtd_final_cic, random_state=42),
    df_cic_misto[df_cic_misto['Label'] == 1].sample(n=qtd_final_cic, random_state=42)
], ignore_index=True)
del df_cic_misto
gc.collect()

for col in features_manter:
    if col not in df_cic_final.columns: df_cic_final[col] = np.float32(0.0)
df_cic_final = df_cic_final[features_manter].copy()

print(f"Salvando arquivo de Treino Bruto em: {path_saida_cic}")
df_cic_final.to_csv(path_saida_cic, index=False)
del df_cic_final
gc.collect()

print("-" * 60)

# ==============================================================================
# --- PARTE 2: PROCESSAMENTO DO UNSW-NB15 (VALIDAÇÃO CRUZADA UDP) ---
# ==============================================================================
print("-> Iniciando auditoria de cabeçalhos do UNSW-NB15...")
cols_train = pd.read_csv(path_unsw_folder + "UNSW_NB15_training-set.csv", nrows=0).columns.str.strip().tolist()

# Função de busca dinâmica contra KeyError
def buscar_coluna_valida(lista_opcoes, lista_disponivel, nome_amigavel):
    for opcao in lista_opcoes:
        if opcao in lista_disponivel:
            return opcao
        for col in lista_disponivel:
            if col.lower() == opcao.lower():
                return col
    return None

# Mapeamento prévio seguro de todas as chaves
col_dur     = buscar_coluna_valida(['dur', 'duration'], cols_train, 'Duração do Fluxo')
col_spkts   = buscar_coluna_valida(['spkts', 's_pkts'], cols_train, 'Pacotes de Ida')
col_dpkts   = buscar_coluna_valida(['dpkts', 'd_pkts'], cols_train, 'Pacotes de Volta')
col_sbytes  = buscar_coluna_valida(['sbytes', 's_bytes'], cols_train, 'Bytes de Ida')
col_dbytes  = buscar_coluna_valida(['dbytes', 'd_bytes'], cols_train, 'Bytes de Volta')
col_max     = buscar_coluna_valida(['Lnk_max', 'max', 'smax', 'dmax'], cols_train, 'Tamanho Máximo')
col_min     = buscar_coluna_valida(['Lnk_min', 'min', 'smin', 'dmin'], cols_train, 'Tamanho Mínimo')
col_smean   = buscar_coluna_valida(['smean', 'smeansz'], cols_train, 'Média de Ida')
col_dmean   = buscar_coluna_valida(['dmean', 'dmeansz'], cols_train, 'Média de Volta')
col_srate   = buscar_coluna_valida(['srate', 'Srate'], cols_train, 'Taxa de Ida')
col_drate   = buscar_coluna_valida(['drate', 'Drate'], cols_train, 'Taxa de Volta')
col_rate    = buscar_coluna_valida(['rate', 'Rate'], cols_train, 'Taxa Total')
col_sinpkt  = buscar_coluna_valida(['sinpkt', 'sintpkt', 'sin_pkt'], cols_train, 'Intervalo Tempo Ida')
col_dinpkt  = buscar_coluna_valida(['dinpkt', 'dintpkt', 'din_pkt'], cols_train, 'Intervalo Tempo Volta')
col_proto   = buscar_coluna_valida(['proto', 'Protocol'], cols_train, 'Protocolo')

print("Carregando e unificando arquivos de dados do UNSW...")
train_unsw = pd.read_csv(path_unsw_folder + "UNSW_NB15_training-set.csv")
test_unsw = pd.read_csv(path_unsw_folder + "UNSW_NB15_testing-set.csv")
df_unsw = pd.concat([train_unsw, test_unsw], ignore_index=True)
del train_unsw, test_unsw
gc.collect()

df_unsw.columns = df_unsw.columns.str.strip()

# Filtragem de Categoria e Protocolo UDP
df_unsw['attack_cat'] = df_unsw['attack_cat'].fillna('Normal').str.strip()
df_unsw = df_unsw[df_unsw['attack_cat'].isin(['Normal', 'DoS'])].copy()

if col_proto:
    df_unsw[col_proto] = df_unsw[col_proto].str.strip().str.lower()
    df_unsw = df_unsw[df_unsw[col_proto] == 'udp'].copy()

df_unsw['Label'] = np.where(df_unsw['attack_cat'] == 'Normal', 0, 1).astype(np.int8)

print("Balanceando classes UDP do UNSW de forma simétrica...")
qtd_ataques = df_unsw[df_unsw['Label'] == 1].shape[0]
df_benignos = df_unsw[df_unsw['Label'] == 0].sample(n=qtd_ataques, random_state=42)
df_ataques = df_unsw[df_unsw['Label'] == 1]

df_unsw_balanced = pd.concat([df_benignos, df_ataques], ignore_index=True)
del df_unsw, df_benignos, df_ataques
gc.collect()

# Criação do DataFrame de Validação final baseado em fluxo
df_validacao = pd.DataFrame()

if col_dur:    df_validacao['Flow Duration'] = df_unsw_balanced[col_dur].astype(np.float32)
if col_spkts:  df_validacao['Total Fwd Packets'] = df_unsw_balanced[col_spkts].astype(np.float32)
if col_dpkts:  df_validacao['Total Backward Packets'] = df_unsw_balanced[col_dpkts].astype(np.float32)
if col_sbytes: df_validacao['Total Length of Fwd Packets'] = df_unsw_balanced[col_sbytes].astype(np.float32)
if col_dbytes: df_validacao['Total Length of Bwd Packets'] = df_unsw_balanced[col_dbytes].astype(np.float32)

if col_max:
    df_validacao['Fwd Packet Length Max'] = df_unsw_balanced[col_max].astype(np.float32)
    df_validacao['Bwd Packet Length Max'] = df_unsw_balanced[col_max].astype(np.float32)
    df_validacao['Max Packet Length'] = df_unsw_balanced[col_max].astype(np.float32)

if col_min:
    df_validacao['Fwd Packet Length Min'] = df_unsw_balanced[col_min].astype(np.float32)
    df_validacao['Bwd Packet Length Min'] = df_unsw_balanced[col_min].astype(np.float32)
    df_validacao['Min Packet Length'] = df_unsw_balanced[col_min].astype(np.float32)

if col_smean: df_validacao['Fwd Packet Length Mean'] = df_unsw_balanced[col_smean].astype(np.float32)
if col_dmean: df_validacao['Bwd Packet Length Mean'] = df_unsw_balanced[col_dmean].astype(np.float32)

if col_srate: df_validacao['Fwd Packets/s'] = df_unsw_balanced[col_srate].astype(np.float32)
if col_drate: df_validacao['Bwd Packets/s'] = df_unsw_balanced[col_drate].astype(np.float32)
if col_rate:  df_validacao['Flow Packets/s'] = df_unsw_balanced[col_rate].astype(np.float32)

if col_sinpkt:
    df_validacao['Flow IAT Mean'] = df_unsw_balanced[col_sinpkt].astype(np.float32)
    df_validacao['Fwd IAT Mean'] = df_unsw_balanced[col_sinpkt].astype(np.float32)
if col_dinpkt:
    df_validacao['Bwd IAT Mean'] = df_unsw_balanced[col_dinpkt].astype(np.float32)

df_validacao['Label'] = df_unsw_balanced['Label']
del df_unsw_balanced
gc.collect()

# Alinhamento final de dimensionalidade (Preenchimento das 59 colunas)
for col in features_manter:
    if col not in df_validacao.columns:
        df_validacao[col] = 0.0

df_validacao = df_validacao[features_manter].copy()

print(f"Salvando arquivo de Validação Bruta (UNSW UDP) em: {path_saida_unsw}")
df_validacao.to_csv(path_saida_unsw, index=False)

print(f"\n--- PROCESSO CONCLUÍDO COM SUCESSO! ---")
print(f"Matriz de Treino (CIC Misto):  {path_saida_cic}")
print(f"Matriz de Validação (UNSW UDP): {path_saida_unsw}")

--- SCRIPT UNIFICADO: GERADOR DE DATASETS (CIC DDOS & UNSW UDP) ---

-> Lendo blocos do arquivo CIC [UDP Puro]...
-> Lendo blocos do arquivo CIC [UDP Lag]...
Unificando e balanceando CIC DDoS...
Salvando arquivo de Treino Bruto em: C:\Users\xrafa\Downloads\archive\CIC_Misto_UDP_Treino.csv
------------------------------------------------------------
-> Iniciando auditoria de cabeçalhos do UNSW-NB15...
Carregando e unificando arquivos de dados do UNSW...
Balanceando classes UDP do UNSW de forma simétrica...
Salvando arquivo de Validação Bruta (UNSW UDP) em: C:\Users\xrafa\Downloads\archive\UNSW_UDP_Validacao.csv

--- PROCESSO CONCLUÍDO COM SUCESSO! ---
Matriz de Treino (CIC Misto):  C:\Users\xrafa\Downloads\archive\CIC_Misto_UDP_Treino.csv
Matriz de Validação (UNSW UDP): C:\Users\xrafa\Downloads\archive\UNSW_UDP_Validacao.csv
